# Week 7.1 — CG vs. Preconditioned CG (Jacobi) on the 1D Poisson Problem
`-u'' = f` on `(0,1)`, `u(0)=u(1)=0`.

In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import cg
from scipy.sparse.linalg import LinearOperator
import matplotlib.pyplot as plt

## Problem setup
Discretizes the 1D Poisson operator with a manufactured solution $u_{\text{exact}}=\sin(\pi x)$ and matching source term $f=\pi^2\sin(\pi x)$ (so that $-u''=f$ is satisfied exactly by $u_{\text{exact}}$), giving a known reference for validating the discretization.

In [2]:
n = 400
h = 1 / (n + 1)
e = np.ones(n)
A = sp.spdiags([-e, 2 * e, -e], [-1, 0, 1], n, n, format='csr') / h**2

# Right-hand side and exact solution for test (sine mode)
xgrid = np.arange(1, n + 1) * h
u_exact = np.sin(np.pi * xgrid)
f = (np.pi**2) * u_exact
b = f

tol = 1e-8
maxit = 1000
x0 = np.zeros(n)

## CG (no preconditioner)
Runs plain CG on the discretized Laplacian, whose condition number grows like $O(n^2)$ with the grid resolution — normally the main motivation for preconditioning.

In [3]:
res_cg = []
def callback_cg(xk):
    res_cg.append(np.linalg.norm(b - A @ xk))

res_cg.append(np.linalg.norm(b - A @ x0))
_, _ = cg(A, b, x0=x0, rtol=tol, maxiter=maxit, callback=callback_cg)

## PCG with Jacobi (diagonal) preconditioner
Applies the cheapest possible preconditioner, $M=\text{diag}(A)$. Note that for this particular discretization every row has the *same* diagonal value $2/h^2$, so $M^{-1}A=\frac{h^2}{2}A$ is just a uniform rescaling of $A$ — it cannot change $A$'s condition number or eigenvalue distribution at all. Jacobi preconditioning only helps when the diagonal *varies* across rows (e.g. non-uniform meshes or coefficients); here it's included mainly to show the mechanics of PCG rather than to demonstrate a speed-up.

In [4]:
M_diag = A.diagonal()
M_inv = lambda r: r / M_diag
M = LinearOperator((n, n), matvec=M_inv)

res_pcg = []
def callback_pcg(xk):
    res_pcg.append(np.linalg.norm(b - A @ xk))

res_pcg.append(np.linalg.norm(b - A @ x0))
_, _ = cg(A, b, x0=x0, rtol=tol, maxiter=maxit, M=M, callback=callback_pcg)

## Compare residual histories
Plots both residual histories; because the diagonal here is constant, expect the two curves to track each other closely rather than showing the usual preconditioning benefit.

In [5]:
plt.semilogy(np.arange(len(res_cg)), res_cg, 'o-', label='CG')
plt.semilogy(np.arange(len(res_pcg)), res_pcg, '*-', label='PCG (Jacobi)')
plt.grid(True)
plt.xlabel('Iteration k')
plt.ylabel('||r_k||_2')
plt.legend(loc='lower left')
plt.title('CG vs PCG (Jacobi) on 1D Laplacian')
plt.show()

/var/folders/k6/1w07pxzj0mx129drg82_k3_w0000gp/T/ipykernel_73699/406117998.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
